En Databricks SQL, las Functions (Funciones) son bloques de lógica reutilizables que encapsulan cálculos, transformaciones o reglas de negocio. En lugar de escribir la misma lógica compleja en diez consultas diferentes, creas una función y la llamas por su nombre.

Desde la llegada de Unity Catalog, las funciones se han vuelto fundamentales porque ahora se pueden compartir entre diferentes usuarios, catálogos y notebooks de forma segura.

**Tipos de Funciones**

- **Scalar Functions (UDFs):** Toman uno o varios valores de entrada y devuelven un único valor. (Ej: Calcular el IVA de un precio).

- **Table-Valued Functions (UDTFs):** Toman parámetros y devuelven una tabla completa. (Ej: Una función que genere una serie de fechas).

- **Built-in Functions:** Las que ya vienen con Databricks (como `abs()`, `date_add()`, `substring()`).

**1. Anatomía de una función (Scalar)**

Para crear una función persistente en Unity Catalog, la sintaxis es muy clara:

In [0]:
%sql
CREATE OR REPLACE FUNCTION catalog.schema.function_name(parameter1 TIPO, parameter2 TIPO)
RETURNS TIPO
[DETERMINISTIC | NOT DETERMINISTIC]
COMMENT 'Descripción de lo que hace'
RETURN lógica_de_la_función;

- `DETERMINISTIC`: Significa que para la misma entrada, siempre dará la misma salida (importante para que Spark optimice la consulta).

- `RETURN`: Aquí es donde vive la "magia" (la fórmula).

#### **Ejemplo:**
Imagina que tienes los precios de los tickets de vuelo en dólares, pero necesitas calcular el precio final con el Impuesto PAIS (30%) y la Retención (30%).

In [0]:
%sql
CREATE OR REPLACE FUNCTION streaming.vuelos.calcular_precio_final(precio_usd DOUBLE)
RETURNS DOUBLE
DETERMINISTIC
COMMENT 'Calcula el precio final de un ticket en Argentina sumando impuestos (60% total)'
RETURN precio_usd * 1.60;

In [0]:
%sql
SELECT 
    id_vuelo, 
    precio_base, 
    streaming.vuelos.calcular_precio_final(precio_base) AS precio_con_impuestos
FROM streaming.vuelos.tickets;

**Row and Column Filters**

Row filters let you **control which rows a user can access in a table based on custom logic**. At query time, a row filter evaluates a condition and returns only the rows that meet it. This is commonly used to implement row-level security—for example, restricting users to records from a specific `region`, `department`, or `account`.

Row filters are defined as SQL user-defined functions (UDFs), and can also incorporate Python or Scala logic when wrapped in a SQL UDF. You can apply row filters per table, or centrally through ABAC policies using governed tags.

**Column Masks**

Column masks **control what values users see in specific columns, depending on who they are**. At query time, the mask replaces each reference to a column with the result of a masking function. This allows sensitive data, such as SSNs or emails, to be redacted or transformed based on user identity or role.

**Each column can have one mask.** The mask must be defined as a SQL UDF that returns a value of the same type as the column being masked. The SQL UDF can optionally [call Python or Scala UDFs](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply#wrapper-example) to implement complex masking logic. Column masks can also take other columns as inputs, for example, to vary behavior based on multiple attributes.

Like row filters, column masks can be applied per table or managed centrally through ABAC policies. They operate at query time and integrate seamlessly with standard SQL, notebooks, and dashboards.

| Feature      | Applies to        | Managed using         | Naming impact              | Best used for...                               |
|--------------|-------------------|-----------------------|----------------------------|-----------------------------------------------|
| Dynamic views | Views            | SQL logic             | Creates a new object name  | Sharing filtered data or spanning multiple tables |
| Row filters   | Tables           | ABAC or mapping tables | Table name unchanged       | Row-level access control tied to user or data tags |
| Column masks  | Tables/columns   | ABAC or mapping tables | Table name unchanged       | Redacting sensitive column data based on identity |

Queremos que `juan.perez@gmail.com` se convierta en `j***@gmail.com`.

In [0]:
%sql
CREATE OR REPLACE FUNCTION streaming.security.mask_email(email STRING)
RETURNS STRING
DETERMINISTIC
RETURN CONCAT(
    SUBSTRING(email, 1, 1), -- Toma la primera letra
    '***@', 
    SPLIT_PART(email, '@', 2) -- Toma todo lo que esté después del @
);

En lugar de dar acceso a la tabla original, le das acceso a los analistas a esta vista:

In [0]:
%sql
CREATE VIEW streaming.vuelos.vw_pasajeros_seguro AS
SELECT 
    id_pasajero,
    streaming.security.mask_email(email) AS email_masked,
    nacionalidad
FROM streaming.vuelos.pasajeros;